# MICrONS Option 3 — Time-resolved PCA trajectories

Per-area, per-stimulus trial-averaged trajectories with the time axis preserved. Resolves the Monet2/Trippy collapse seen in Option 2 by keeping within-trial dynamics. See `docs/specs/2026-05-04-option3-trajectories-design.md`.

**Sections will be filled in by subsequent tasks.**

In [ ]:
# === Smoke test: full module (sections a-d) on real session 7_5 data ===
# This cell is REPLACED by proper Part 0 in Task 3.
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import microns_eda
import option2_pca_utils
import option3_trajectories_utils as traj_utils

DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
SESSION = "7_5"
RANDOM_SEED = 42

# (Identical setup to the previous smoke test — kept compact here.)
reader = microns_eda.open_dataset(DATADIR)
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION
)
meta = microns_eda.get_session_meta(DATADIR, SESSION)
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, _ = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=1.0
)
stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_full = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_full[clean_trial_indices]
responses_pp = option2_pca_utils.preprocess_responses(responses, apply_log=False)

# Trajectories + PCA + distances on V1.
trajectories = traj_utils.build_stim_trajectories(
    responses_pp, trial_boundaries, clean_trial_indices,
    meta["brain_areas"], labels, n_frames=75,
)
pca_per_area = traj_utils.fit_trajectory_pca(trajectories, n_components=10)

# Plotters: smoke-test five (PSTH + 2D + distance time course + cross-area-headline + Clip-sub).
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
traj_utils.plot_psth_per_stim_area(trajectories, "V1", ax=axes[0, 0])
traj_utils.plot_trajectory_2d(
    pca_per_area["V1"]["stim_pcs"], "V1", pca_per_area["V1"]["pca"],
    ax=axes[0, 1],
)
v1_d_full = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="full"
)
traj_utils.plot_pairwise_distance_time_course(
    v1_d_full, None, "V1", "Euclidean (full features)", ax=axes[1, 0],
)
# Cross-area Monet2-Trippy.
m2t = {
    area: traj_utils.pairwise_trajectory_distance(
        trajectories[area], metric="full"
    )[frozenset({"Monet2", "Trippy"})]
    for area in trajectories
}
traj_utils.plot_cross_area_monet2_trippy(
    m2t, None, "Euclidean (full features)", ax=axes[1, 1],
)
plt.tight_layout()
plt.show()

print("plotter smoke test OK — 4 panels rendered, no errors")